# KanbanColumnBoard 中等复杂度迁移实验：失败分析与评估链路修复

- 实验日期：2026-09-17
- 修复与记录日期：2026-09-21
- 输入：`data/datasets/components/02_medium/KanbanColumnBoard/vue/KanbanColumnBoard.vue`
- 原始报告：`data/experiments/repair_reports/KanbanColumnBoard_repair_history.json`
- 目的：验证 simple 样本通过后，pipeline 能否处理 computed、嵌套循环、props 默认值和带参数事件。


## 1. 原始实验结果

原始 pipeline 连续执行三轮 repair 后仍失败：

| 指标 | 初始 | Repair 1 | Repair 2 | Repair 3 / 最终 |
|---|---:|---:|---:|---:|
| 静态错误 | 0 | 0 | 0 | 0 |
| 结构相似度 | 0.4688 | 0.4688 | 0.4688 | 0.4688 |
| 文本相似度 | 0.6562 | 0.6562 | 0.6562 | 0.6562 |
| Vue/San 节点数 | 16/13 | 16/13 | 16/13 | 16/13 |

最终状态为 `max_repair_rounds_reached`。三轮 repair 共消耗约 60,460 tokens，但结构指标没有改善。该现象提示反馈信号可能不可靠，而不只是模型没有修改代码。


In [ ]:
import json
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'migration_pipeline').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

root = find_repo_root()
report_path = root / 'data/experiments/repair_reports/KanbanColumnBoard_repair_history.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
history = report['repair_history']
final_rerun_metrics = {
    'final_passed': report['summary']['final_passed'],
    'stop_reason': report['summary']['stop_reason'],
    'repair_rounds': report['summary']['repair_rounds'],
    'initial_structure_similarity': report['initial']['visual_eval']['structure_similarity'],
    'final_structure_similarity': report['final']['visual_eval']['structure_similarity'],
    'repair_total_tokens': sum(item['repair_usage']['total_tokens'] for item in history),
}
assert final_rerun_metrics['repair_rounds'] == 1
assert final_rerun_metrics['final_passed'] is True
assert final_rerun_metrics['initial_structure_similarity'] == 0.2105
assert final_rerun_metrics['final_structure_similarity'] == 1.0
assert final_rerun_metrics['repair_total_tokens'] == 19425
final_rerun_metrics


## 2. 根因分层

### 2.1 评估器缺陷

原 San renderer 没有执行组件顶层方法，无法运行参考实现的 `this.setValue(...)`；没有计算 computed；把 `s-for` 直接删除而非展开；嵌套循环的 `column`、`task` 局部变量没有作用域。Vue renderer 虽能解析 props 默认值，却没有执行 `created()`。因此原始快照出现以下异常：

- Vue 默认任务未进入 `tasks`，三列计数均为 0；
- San 只生成一个空 article，并报告 `selectedTask/column/task is not defined`；
- 人工验证过的参考 San 也无法通过同一评估器。

所以原始 `structure_similarity=0.4688` 属于混合信号，不能直接解释为生成组件的真实结构质量。

### 2.2 生成结果的真实缺陷

生成代码也存在独立于评估器的缺陷：

1. SSM 已保存 `initialTasks` 和 `columns` 的 Vue prop 默认值，但生成的 `initData()` 没有落地这两个字段；无外部传参时 San 看板为空。
2. `move(-1)`、`move(1)`、`select(task.id)` 被生成成无参数的 `move`、`select`。后续 repair 又改成 `data-args/getAttribute` 绕行，没有恢复直接的 San 事件表达式。
3. 原 validator 只检查 handler 是否存在，没有检查默认值和事件参数是否完整。


## 3. 按步骤实施的修复

### 步骤一：修复 Vue 轻量 renderer

- 在 `data()` 与初次 computed 后执行 `created()`；
- 生命周期改变状态后重新计算 computed；
- 嵌套 `v-for` 按局部作用域递归渲染；
- `:key` 仅作为虚拟 DOM 身份，不进入最终 DOM 快照。

### 步骤二：修复 San 轻量 renderer

- 建立包含 `data/watch/fire` 和组件顶层方法的执行上下文；
- 执行 `inited()`、`attached()` 后计算 computed；
- 支持嵌套 `s-for`、循环局部变量、动态 class/value/disabled；
- 移除 `key/trackby` 等非真实 DOM 属性。

### 步骤三：增强 generation/repair prompt

- Vue props 的默认值必须作为同名字段落入 San `initData()`；
- 带参数事件必须直接保留为 `on-click="handler(arg)"`；
- 禁止丢失参数以及使用 `data-args/getAttribute` 绕行。

### 步骤四：增强 validator

新增 `prop_defaults_in_init_data`、`event_arguments_consistency` 和 `custom_event_migration` 三项检查。缺陷结果现在会报告：

- `missing_prop_defaults`；
- `event_arguments_not_landed`；
- `custom_event_not_fire`（Vue `$emit` 被错误迁移为 San `this.dispatch`）。


In [ ]:
import sys
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from SSM.extractors.factory import SSMFactory
from migration_pipeline.stages.validate import ValidateStage, ValidateStageInput
from migration_pipeline.stages.visual_eval import VisualEvalStage, VisualEvalStageInput

vue_path = root / 'data/datasets/components/02_medium/KanbanColumnBoard/vue/KanbanColumnBoard.vue'
reference_path = root / 'data/datasets/components/02_medium/KanbanColumnBoard/san/KanbanColumnBoard.san'
repaired_path = root / 'data/experiments/pipeline/KanbanColumnBoard_repaired.san'
ssm = SSMFactory().build_from_file(str(vue_path))

reference_visual = VisualEvalStage().run(VisualEvalStageInput(
    vue_file_path=str(vue_path),
    generated_file_path=str(reference_path),
))
reference_validation = ValidateStage().run(ValidateStageInput(
    generated_file_path=str(reference_path),
    ssm=ssm,
))
repaired_validation = ValidateStage().run(ValidateStageInput(
    generated_file_path=str(repaired_path),
    ssm=ssm,
))
repaired_codes = {item['code'] for item in repaired_validation.validation_errors}
repaired_code = repaired_path.read_text(encoding='utf-8')

assert reference_visual.visual_eval_passed
assert reference_visual.visual_eval_warnings == []
assert reference_visual.vue_render_result['dom_snapshot']['node_count'] == 19
assert reference_visual.san_render_result['dom_snapshot']['node_count'] == 19
assert reference_visual.structure_similarity == 1.0
assert reference_visual.tag_sequence_similarity == 1.0
assert reference_visual.text_similarity == 1.0
assert reference_validation.validation_passed
assert repaired_validation.validation_passed
assert repaired_codes == set()
assert 'this.fire(' in repaired_code
assert 'this.dispatch(' not in repaired_code
{
    'reference_nodes': (19, 19),
    'reference_similarity': (1.0, 1.0, 1.0),
    'final_rerun_validation_errors': sorted(repaired_codes),
}


## 4. 修复验证结果与证据边界

离线回归测试 `tests/test_migration_pipeline_regressions.py` 包含四项检查：

1. 参考 Vue/San 初始渲染均为 19 个节点，renderer 无 diagnostics；
2. 结构、标签序列、文本相似度均为 1.0；
3. 新 validator 能识别默认值与事件参数缺失，同时放行参考实现；
4. 新增负向测试，确保 Vue `$emit` 不能被迁移为 San `this.dispatch`。

第一次完整重跑在 1 轮 repair 后由报告判定通过：Vue/San 节点数由 19/10 修复为 19/19，结构、标签序列和文本相似度均达到 1.0，共消耗 19,345 repair tokens。随后进行源码复核时发现最终代码仍使用 `this.dispatch('move', ...)` 和 `this.dispatch('add', ...)`。该 API 用于组件消息派发，不等价于 Vue `$emit` 对应的 San 自定义事件 `this.fire(...)`。

第二次完整重跑加载了 `custom_event_not_fire` 校验与更新后的 prompt。初始生成已使用 `this.fire(name, payload)`；经过 1 轮 repair 后补齐 props 默认值，最终代码直接使用 `this.fire('move', ...)` 和 `this.fire('add', ...)`。最终静态校验为 0 errors / 0 warnings，Vue/San 均为 19 个节点，structure/tag/text similarity 均为 1.0，本轮 repair 消耗 19,425 tokens。

> 结论边界：最终结果通过静态契约校验、San 编译检查与项目轻量 renderer 对比；尚未将其表述为真实浏览器自动化交互验证。


## 5. 第二次完整重跑结果

重启包含最新 `local_server/api/evaluation_routes.py` 的服务后执行了：

```powershell
python migration_pipeline/run_pipeline.py `
  --vue-file data/datasets/components/02_medium/KanbanColumnBoard/vue/KanbanColumnBoard.vue `
  --output-file data/experiments/pipeline/KanbanColumnBoard.san `
  --max-repair-rounds 3 `
  --include-state
```

验收结果：

- `final_passed = true`，`stop_reason = passed`，repair 轮数为 1；
- validation 为 0 errors / 0 warnings；
- Vue/San 节点数均为 19，tree edit distance 为 0；
- structure/tag/text similarity 均为 1.0；
- `initData()` 包含 `initialTasks` 和 `columns`；
- template 直接包含 `move(-1)`、`move(1)`、`select(task.id)`；
- 最终脚本包含 `this.fire(...)`，且不包含 `this.dispatch(...)`。


## 6. 研究结论

本实验表明，迁移闭环的 repair 效果高度依赖评估反馈的真实性。轻量 renderer 对生命周期、computed 和嵌套循环支持不足时，会把正确参考实现判为失败，并驱动模型针对错误信号进行重复修复，造成 token 消耗而指标不变。同时，单纯检查 San 语法、handler 名称和初始 DOM 等价性，不足以发现 props 默认值丢失、事件参数丢失以及 `fire/dispatch` API 语义混淆。

因此后续批量实验应先用人工验证过的参考组件校准 evaluator，再运行生成结果；对每类复杂语法至少建立一个正向参考用例和一个负向缺陷用例。该策略能把“模型生成失败”和“评估器假阴性”分离，避免修复闭环优化错误目标。


## 7. 文件索引

- 原始报告：`data/experiments/repair_reports/KanbanColumnBoard_repair_history.json`
- 原始与 repair 输出：`data/experiments/pipeline/KanbanColumnBoard*.san`
- Vue renderer：`migration_pipeline/utils/vue_render.py`
- San renderer：`migration_pipeline/utils/san_render.py`
- Validator：`migration_pipeline/stages/validate.py`
- Generation prompt：`local_server/api/evaluation_routes.py`
- Repair prompt：`migration_pipeline/utils/repair_prompt.py`
- 回归测试：`tests/test_migration_pipeline_regressions.py`
